In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("datasets/train.csv")

In [5]:
df.head()

,QuestionId,ConstructId,ConstructName,SubjectId,SubjectName,CorrectAnswer,QuestionText,AnswerAText,AnswerBText,AnswerCText,AnswerDText,MisconceptionAId,MisconceptionBId,MisconceptionCId,MisconceptionDId
0,0,856,Use the order of operations to carry out calcu...,33,BIDMAS,A,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \),\( 3 \times(2+4-5) \),Does not need brackets,NaN,NaN,NaN,1672.0
1,1,1612,Simplify an algebraic fraction by factorising ...,1077,Simplifying Algebraic Fractions,D,"Simplify the following, if possible: \( \frac{...",\( m+1 \),\( m+2 \),\( m-1 \),Does not simplify,2142.0,143.0,2142.0,NaN
2,2,2774,Calculate the range from a list of data,339,Range and Interquartile Range from a List of Data,B,Tom and Katie are discussing the \( 5 \) plant...,Only\nTom,Only\nKatie,Both Tom and Katie,Neither is correct,1287.0,NaN,1287.0,1073.0
3,3,2377,Recall and use the intersecting diagonals prop...,88,Properties of Quadrilaterals,C,The angles highlighted on this rectangle with ...,acute,obtuse,\( 90^{\circ} \),Not enough information,1180.0,1180.0,NaN,1180.0
4,4,3387,Substitute positive integer values into formul...,67,Substitution into Formula,A,The equation \( f=3 r^{2}+3 \) is used to find...,\( 30 \),\( 27 \),\( 51 \),\( 24 \),NaN,NaN,NaN,1818.0


In [8]:
df.isnull().sum()

QuestionId            0
ConstructId           0
ConstructName         0
SubjectId             0
SubjectName           0
CorrectAnswer         0
QuestionText          0
AnswerAText           0
AnswerBText           0
AnswerCText           0
AnswerDText           0
MisconceptionAId    734
MisconceptionBId    751
MisconceptionCId    789
MisconceptionDId    832
dtype: int64

In [6]:
print("Total questions:", df['QuestionId'].nunique())
print("Total subjects:", df['SubjectName'].nunique())
print("Total constructs:", df['ConstructName'].nunique())

Total questions: 1869
Total subjects: 163
Total constructs: 757


In [10]:
misconception_cols = ['MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']
all_misconceptions = pd.concat([df[c] for c in misconception_cols]).dropna()

In [11]:
freq = all_misconceptions.value_counts()
print(freq.describe())

count    1604.000000
mean        2.724439
std         3.598555
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        54.000000
Name: count, dtype: float64


In [28]:
print("Misconceptions with <20 examples:", (freq < 20).sum(), "out of", len(freq))

Misconceptions with <20 examples: 1592 out of 1604


In [29]:
print("% of total misconception instances covered by those <20:", all_misconceptions[all_misconceptions.isin(freq[freq < 20].index)].shape[0] / len(all_misconceptions) * 100)

% of total misconception instances covered by those <20: 91.39588100686498


In [15]:
no_misconception = df[misconception_cols].isna().all(axis=1)
print("Questions with no misconception tags at all:", no_misconception.sum())

Questions with no misconception tags at all: 0


In [25]:
top_misconceptions = freq[freq >= 20]
print(top_misconceptions)

1214.0    54
1379.0    43
2316.0    38
1507.0    36
1990.0    33
1880.0    32
1597.0    27
2392.0    27
220.0     22
77.0      22
1248.0    22
1072.0    20
Name: count, dtype: int64


In [27]:
relevant_ids = top_misconceptions.index
relevant_ids

Index([1214.0, 1379.0, 2316.0, 1507.0, 1990.0, 1880.0, 1597.0, 2392.0,  220.0,
         77.0, 1248.0, 1072.0],
      dtype='float64')

In [30]:
mask = df[misconception_cols].isin(relevant_ids).any(axis=1)
subset = df[mask]
print(subset['SubjectName'].value_counts())
print(subset['ConstructName'].value_counts())

SubjectName
BIDMAS                                                            26
Linear Equations                                                  25
Rounding to the Nearest Whole (10, 100, etc)                      19
Function Machines                                                 17
Reflection                                                        14
Rounding to Decimal Places                                        14
Expanding Single Brackets                                         12
Inequalities on Number Lines                                      11
Squares, Cubes, etc                                               10
Estimation                                                        10
Solving Linear Inequalities                                        9
Writing Expressions                                                9
Expanding Double Brackets                                          7
Rounding to Significant Figures                                    7
Plotting Quadratics fr

In [32]:
print(subset.shape)  # how many rows total?
print(subset[misconception_cols].head(10)) 

(279, 15)
    MisconceptionAId  MisconceptionBId  MisconceptionCId  MisconceptionDId
14            1775.0            1248.0            1529.0               NaN
26               NaN            2474.0            2316.0            1944.0
27               NaN            1597.0               NaN               NaN
29            1214.0               NaN             811.0            1214.0
30               NaN               NaN             220.0               NaN
45             220.0            1911.0            2085.0               NaN
58            1880.0               NaN               NaN             550.0
65               NaN            2316.0            2245.0               NaN
67               NaN            2248.0            1012.0            2316.0
73            1988.0               NaN            1248.0               NaN


In [33]:
for mid in relevant_ids:
    rows = df[(df[misconception_cols] == mid).any(axis=1)]
    print(f"Misconception {mid}: {rows['ConstructName'].nunique()} unique constructs, top: {rows['ConstructName'].value_counts().head(3).to_dict()}")

Misconception 1214.0: 19 unique constructs, top: {'Solve two-step linear equations, with the variable on one side, with all positive integers': 5, 'Solve two-step linear equations, with the variable on one side, involving positive fractions': 4, 'Solve linear equations with the variable appearing on both sides, with all positive integers': 4}
Misconception 1379.0: 17 unique constructs, top: {'Round numbers to one decimal place': 5, 'Round non-integers to the nearest 10': 5, 'Round numbers to two decimal places': 4}
Misconception 2316.0: 22 unique constructs, top: {'Calculate the square of a number': 10, 'Solve positive quadratic inequalities by balancing e.g x² ≤ a': 2, 'Given a positive x value, find the corresponding y value for curves in the form y = x² + c': 2}
Misconception 1507.0: 15 unique constructs, top: {'Use the order of operations to carry out calculations involving addition, subtraction, multiplication, and/or division': 9, 'Use the order of operations to carry out calcula